# 09 - Random Seed Comparison

Train the EVA unfrozen ArcFace model with the default hyperparameters and compare 10 different random seeds.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.wandb_utils as wandb_utils
from src.models import ArcFaceModel
from src.training import build_optimizer, fit
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

device = get_device()
device

## Config

In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("/sc/projects/sci-zacharatou/mpws2025ez1/wan/checkpoints/e14_seed_comparison"),
    "results_dir": Path("../checkpoints/e09_seed_comparison"),
    "results_path": Path("../checkpoints/e09_seed_comparison/seed_results.csv"),

    # Backbone
    "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
    "input_size": 448,
    "freeze_backbone": False,
    "train_last_n_layers": 0,

    # Model defaults
    "embedding_dim": 256,
    "hidden_dim": 512,
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training defaults
    "batch_size": 16,
    "learning_rate": 1e-4,
    "head_learning_rate": 1e-4,
    "backbone_learning_rate": 1e-5,
    "weight_decay": 1e-5,
    "num_epochs": 25,
    "patience": 8,
    "scheduler_patience": 2,
    "val_split": 0.2,
    "num_workers": 2,
    "augment_train": True,

    # Seeds
    "seed_values": list(range(42, 52)),

    # Image normalization
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
}

config["rerank"] = {
    "enabled": True,
    "k1": 20,
    "k2": 6,
    "lambda_value": 0.3,
}

config["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
config["results_dir"].mkdir(parents=True, exist_ok=True)
config

## Helpers

In [ ]:
def load_data(run_config, backbone):
    train_df = data.load_train_df(run_config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    train_data, val_data = data.train_val_split(
        train_df,
        val_split=run_config["val_split"],
        seed=run_config["seed"],
        stratify_col="ground_truth",
    )

    train_loader, val_loader = data.create_dataloaders(
        train_data,
        val_data,
        img_dir=run_config["data_dir"] / "train" / "train",
        input_size=run_config["input_size"],
        batch_size=run_config["batch_size"],
        num_workers=run_config["num_workers"],
        mean=run_config["mean"],
        std=run_config["std"],
        weighted_sampling=False,
        augment=run_config.get("augment_train", False),
    )

    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    return {
        "label_encoder": label_encoder,
        "num_classes": num_classes,
        "train_loader": train_loader,
        "val_loader": val_loader,
    }


def build_run_config(base_config, seed):
    run_config = base_config.copy()
    run_config["seed"] = seed
    run_config["name"] = f"eva_unfrozen_rs_04_seed_{seed:02d}_hlr1e-04_blr1e-05_wd1e-05_do0.2_aug1_bs16"
    run_config["learning_rate"] = run_config["head_learning_rate"]
    return run_config


## Run Seed Comparison

In [ ]:
def run_experiment(run_config, run_idx, total_runs):
    print("=" * 80)
    print(f"Run {run_idx}/{total_runs}: {run_config['name']}")
    print({
        "seed": run_config["seed"],
        "head_learning_rate": run_config["head_learning_rate"],
        "backbone_learning_rate": run_config["backbone_learning_rate"],
        "weight_decay": run_config["weight_decay"],
        "dropout": run_config["dropout"],
        "augment_train": run_config["augment_train"],
        "batch_size": run_config["batch_size"],
    })

    set_seed(run_config["seed"])

    train_df = data.load_train_df(run_config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    model = ArcFaceModel(
        backbone_name=run_config["backbone_name"],
        num_classes=num_classes,
        embedding_dim=run_config["embedding_dim"],
        hidden_dim=run_config["hidden_dim"],
        margin=run_config["arcface_margin"],
        scale=run_config["arcface_scale"],
        dropout=run_config["dropout"],
        pretrained=True,
        freeze_backbone=False,
        train_last_n_layers=run_config["train_last_n_layers"],
    ).to(device)

    data_bundle = load_data(run_config, model.backbone)
    label_encoder = data_bundle["label_encoder"]
    train_loader = data_bundle["train_loader"]
    val_loader = data_bundle["val_loader"]

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone_params = sum(p.numel() for p in model.backbone.parameters())

    wandb = wandb_utils.init_wandb(
        run_config,
        run_name=run_config["name"],
        param_count=trainable_params,
        param_count_backbone=backbone_params,
    )
    wandb.run.summary["total_param_count"] = total_params
    wandb.run.summary["trainable_param_count"] = trainable_params
    wandb.run.summary["seed_run_index"] = run_idx
    wandb.run.summary["seed_run_total"] = total_runs

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, run_config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=run_config["scheduler_patience"],
    )

    checkpoint_name = f"{run_config['name']}.pth"
    checkpoint_path = run_config["checkpoint_dir"] / checkpoint_name
    results = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=run_config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_mAP_rerank"] = results.get("best_map_rerank")
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    wandb_utils.log_checkpoint_artifact(
        wandb,
        checkpoint_path,
        artifact_name=run_config["name"],
        description="seed-comparison checkpoint",
    )

    wandb.finish()

    return {
        "name": run_config["name"],
        "seed": run_config["seed"],
        "best_val_loss": results["best_val_loss"],
        "best_map": results["best_map"],
        "best_map_rerank": results.get("best_map_rerank"),
        "best_epoch": results["best_epoch"],
        "epochs_ran": results["epochs_ran"],
        "checkpoint_path": str(checkpoint_path),
    }


results = []
seed_values = config["seed_values"]

for run_idx, seed in enumerate(seed_values, start=1):
    run_config = build_run_config(config, seed)
    result = run_experiment(run_config, run_idx, len(seed_values))
    results.append(result)

results_df = pd.DataFrame(results).sort_values(["best_map_rerank", "best_map"], ascending=False)
results_df.to_csv(config["results_path"], index=False)
results_df

## Summary

In [ ]:
summary_df = pd.DataFrame(
    {
        "metric": ["best_map", "best_map_rerank", "best_val_loss", "best_epoch", "epochs_ran"],
        "mean": [
            results_df["best_map"].mean(),
            results_df["best_map_rerank"].mean(),
            results_df["best_val_loss"].mean(),
            results_df["best_epoch"].mean(),
            results_df["epochs_ran"].mean(),
        ],
        "std": [
            results_df["best_map"].std(),
            results_df["best_map_rerank"].std(),
            results_df["best_val_loss"].std(),
            results_df["best_epoch"].std(),
            results_df["epochs_ran"].std(),
        ],
    }
)

results_df[["name", "seed", "best_map", "best_map_rerank", "best_val_loss", "best_epoch", "epochs_ran"]]

In [ ]:
summary_df

View the runs on W&B: [W&B Run Group](https://wandb.ai/juggling-jaguars/jaguar-reid-jugglingjaguars/groups/Experiment-9-RandomSeeds)

![](../images/e9_wandb_dashboard.png)